# Rubidium SOC Generator - 500K Pairs

Genera 500,000 pares de conversación sintética (U:/B:) usando metodología SOC.
Divide en 30 archivos para entrenamiento distribuido.

**Hardware**: Kaggle GPU P100 (30h/sem) para generación con Phi-3-mini
**Output**: 30 archivos `chat_06.txt` a `chat_35.txt` + formatos JSONL/ShareGPT

In [ ]:
# ============================================================
# INSTALACIÓN DE DEPENDENCIAS
# ============================================================
import subprocess, sys, json, os, asyncio, aiohttp, random, hashlib
from pathlib import Path
from tqdm.notebook import tqdm

# Instalar dependencias
!pip install -q aiohttp tqdm transformers accelerate bitsandbytes 2>/dev/null

# Verificar GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# ============================================================
# CONFIGURACIÓN SOC
# ============================================================

class SOCConfig:
    # Modelo generador (usar Ollama local o API)
    generator_model: str = "phi3:mini"  # ollama model
    api_base: str = "http://localhost:11434"
    
    # Target
    target_pairs: int = 500000
    batch_size: int = 16
    max_turns: int = 8
    min_turns: int = 3
    temperature: float = 0.8
    top_p: float = 0.9
    
    # Output
    output_dir: str = "/kaggle/working/soc_corpus"
    checkpoint_every: int = 10000
    num_output_files: int = 30
    
    # Personas y temas
    USER_PERSONAS = {
        "estudiante": {
            "style": "informal, curioso, a veces impreciso",
            "topics": ["tareas", "examenes", "proyectos", "becas", "universidad"],
            "vocab": ["profe", "parcial", "tp", "grupo", "bibliografia"]
        },
        "profesional_tech": {
            "style": "técnico, preciso, usa terminología",
            "topics": ["arquitectura", "deploy", "debugging", "performance", "ci/cd"],
            "vocab": ["latencia", "throughput", "bottleneck", "refactor", "legacy"]
        },
        "curioso_general": {
            "style": "exploratorio, pregunta seguidas, entusiasta",
            "topics": ["ciencia", "historia", "cultura", "tecnologia", "vida"],
            "vocab": ["interesante", "nunca supe", "cómo es que", "qué tal si"]
        },
        "usuario_frustrado": {
            "style": "directo, problema urgente, poco contexto",
            "topics": ["error", "no funciona", "fallo", "bug", "urgente"],
            "vocab": ["nada funciona", "ya probé", "error 500", "timeout", "crash"]
        },
        "creativo": {
            "style": "imaginativo, pide formatos específicos",
            "topics": ["historias", "poemas", "guiones", "ideas", "worldbuilding"],
            "vocab": ["escribe", "inventa", "crea", "imagina", "personaje"]
        },
        "docente_tutor": {
            "style": "pedagógico, explica paso a paso, paciente",
            "topics": ["explicación", "ejemplo", "ejercicio", "concepto", "duda"],
            "vocab": ["veamos", "paso a paso", "por ejemplo", "la idea es", "entonces"]
        },
        "usuario_casual": {
            "style": "cotidiano, abreviado, coloquial",
            "topics": ["planes", "comida", "series", "música", "fin de semana"],
            "vocab": ["qué tal", "anda", "dale", "copado", "tranqui"]
        },
        "experto_dominio": {
            "style": "profundo, técnico, referencias precisas",
            "topics": ["paper", "arquitectura", "optimización", "teoría", "avanzado"],
            "vocab": ["según", "paper", "benchmark", "state-of-the-art", "baseline"]
        },
    }

    TOPICS_ES = [
        "programacion_python", "programacion_javascript", "web_dev", "machine_learning",
        "devops_cloud", "bases_datos", "seguridad_informatica", "movil_android",
        "ciencia_datos", "inteligencia_artificial", "matematicas", "fisica",
        "biologia", "quimica", "astronomia", "neurociencia",
        "psicologia", "filosofia", "historia", "economia",
        "finanzas_personales", "emprendimiento", "marketing", "diseno_ux",
        "salud_mental", "ejercicio", "nutricion", "meditacion",
        "relaciones", "comunicacion", "liderazgo", "productividad",
        "cocina", "viajes", "fotografia", "musica", "literatura",
        "videojuegos", "cine_series", "anime_manga", "tecnologia_consumidor",
        "automoviles", "bricolaje", "jardineria", "mascotas",
        "educacion", "idiomas", "certificaciones", "carrera_profesional"
    ]

    SYSTEM_PROMPT = """Eres un generador de conversaciones realistas en español.
Genera diálogos naturales entre un USUARIO y un ASISTENTE.

REGLAS:
1. El USUARIO tiene una persona específica ({persona})
2. El ASISTENTE es útil, honesto, y admite cuando no sabe
3. Conversación de {min_turns}-{max_turns} turnos
4. Incluye hesitación natural, correcciones, referencias previas
5. Tema: {topic}
6. Español neutro/latinoamericano, natural
7. Longitud variable: respuestas cortas (1-2 frases) y largas (párrafos)

FORMATO DE SALIDA (JSON):
{{
  "persona": "nombre_persona",
  "topic": "tema",
  "turns": [
    {{"role": "user", "content": "..."}},
    {{"role": "assistant", "content": "..."}},
    ...
  ]
}}"""

    USER_PROMPT_TEMPLATE = """Genera una conversación para:
- Persona: {persona} ({style})
- Tema: {topic}
- Vocabulario típico: {vocab}
- Turnos: {min_turns}-{max_turns}

La conversación debe sentirse REAL, como si la hubieras extraído de un foro/chat real."""

CONFIG = SOCConfig()
print(f"Config: {CONFIG.target_pairs} pares, {CONFIG.num_output_files} archivos")
print(f"Personas: {len(CONFIG.USER_PERSONAS)}, Temas: {len(CONFIG.TOPICS_ES)}")

In [ ]:
# ============================================================
# GENERADOR SOC CON OLLAMA (Phi-3-mini local)
# ============================================================

# Instalar Ollama y Phi-3-mini
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
!ollama serve > /tmp/ollama.log 2>&1 &
import time; time.sleep(5)
!ollama pull phi3:mini

# Verificar
!ollama list

In [ ]:
import asyncio
import aiohttp
import json
import random
import hashlib
from pathlib import Path
from tqdm.notebook import tqdm

class SOCGenerator:
    def __init__(self, config):
        self.config = config
        self.session = None
        self.generated = 0
        self.seen_hashes = set()
        self.all_conversations = []
        
    async def __aenter__(self):
        self.session = aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=120))
        return self
        
    async def __aexit__(self, *args):
        if self.session:
            await self.session.close()
    
    def _hash_conversation(self, turns):
        text = "".join(t["content"] for t in turns)
        return hashlib.md5(text.encode()).hexdigest()[:16]
    
    def _is_quality(self, turns):
        if len(turns) < self.config.min_turns * 2:
            return False
        for t in turns:
            if len(t["content"]) < 15:
                return False
        user_texts = [t["content"] for t in turns if t["role"] == "user"]
        if len(set(user_texts)) != len(user_texts):
            return False
        return True
    
    async def generate_batch(self, batch_size):
        tasks = []
        for _ in range(batch_size):
            persona_key = random.choice(list(self.config.USER_PERSONAS.keys()))
            persona = self.config.USER_PERSONAS[persona_key]
            topic = random.choice(self.config.TOPICS_ES)
            
            prompt = self.config.USER_PROMPT_TEMPLATE.format(
                persona=persona_key,
                style=persona["style"],
                topic=topic,
                vocab=", ".join(persona["vocab"]),
                min_turns=self.config.min_turns,
                max_turns=self.config.max_turns
            )
            
            tasks.append(self._generate_one(persona_key, topic, prompt))
        
        results = await asyncio.gather(*tasks, return_exceptions=True)
        valid = []
        for r in results:
            if isinstance(r, dict) and "turns" in r and self._is_quality(r["turns"]):
                h = self._hash_conversation(r["turns"])
                if h not in self.seen_hashes:
                    self.seen_hashes.add(h)
                    valid.append(r)
        return valid
    
    async def _generate_one(self, persona, topic, prompt):
        system = self.config.SYSTEM_PROMPT.format(
            persona=persona,
            min_turns=self.config.min_turns,
            max_turns=self.config.max_turns,
            topic=topic
        )
        
        payload = {
            "model": self.config.generator_model,
            "prompt": f"{system}\n\n{prompt}",
            "temperature": self.config.temperature,
            "top_p": self.config.top_p,
            "format": "json",
            "stream": False,
            "options": {"num_predict": 2048}
        }
        
        try:
            async with self.session.post(f"{self.config.api_base}/api/generate", json=payload) as resp:
                data = await resp.json()
                response_text = data.get("response", "{}")
            
            conv = json.loads(response_text)
            conv["persona"] = persona
            conv["topic"] = topic
            return conv
        except Exception as e:
            return {"error": str(e)}
    
    def save_checkpoint(self, conversations, suffix):
        Path(self.config.output_dir).mkdir(parents=True, exist_ok=True)
        
        # JSONL
        with open(f"{self.config.output_dir}/soc_{suffix}.jsonl", "w", encoding="utf-8") as f:
            for c in conversations:
                f.write(json.dumps(c, ensure_ascii=False) + "\n")
        
        # U:/B: format
        with open(f"{self.config.output_dir}/soc_{suffix}.txt", "w", encoding="utf-8") as f:
            for c in conversations:
                for turn in c["turns"]:
                    role = "U" if turn["role"] == "user" else "B"
                    f.write(f"{role}: {turn['content']}\n")
                f.write("\n")
        
        # ShareGPT format
        sharegpt = []
        for c in conversations:
            conv = []
            for turn in c["turns"]:
                conv.append({
                    "from": "human" if turn["role"] == "user" else "gpt",
                    "value": turn["content"]
                })
            sharegpt.append({"conversations": conv})
        
        with open(f"{self.config.output_dir}/soc_{suffix}_sharegpt.json", "w", encoding="utf-8") as f:
            json.dump(sharegpt, f, ensure_ascii=False, indent=2)
    
    def split_into_files(self, conversations):
        """Divide conversaciones en N archivos balanceados"""
        pairs_per_file = len(conversations) // self.config.num_output_files
        
        for i in range(self.config.num_output_files):
            start = i * pairs_per_file
            end = start + pairs_per_file if i < self.config.num_output_files - 1 else len(conversations)
            batch = conversations[start:end]
            
            # chat_06.txt a chat_35.txt (30 archivos)
            file_num = 6 + i
            filename = f"chat_{file_num:02d}.txt"
            
            with open(f"{self.config.output_dir}/{filename}", "w", encoding="utf-8") as f:
                for c in batch:
                    for turn in c["turns"]:
                        role = "U" if turn["role"] == "user" else "B"
                        f.write(f"{role}: {turn['content']}\n")
                    f.write("\n")
            
            print(f"  {filename}: {len(batch)} conversaciones")
        
        # También guardar JSONL consolidado
        with open(f"{self.config.output_dir}/soc_all.jsonl", "w", encoding="utf-8") as f:
            for c in conversations:
                f.write(json.dumps(c, ensure_ascii=False) + "\n")


In [ ]:
# ============================================================
# EJECUCIÓN PRINCIPAL
# ============================================================

async def main():
    async with SOCGenerator(CONFIG) as gen:
        pbar = tqdm(total=CONFIG.target_pairs, desc="Generando SOC")
        
        while gen.generated < CONFIG.target_pairs:
            batch = await gen.generate_batch(CONFIG.batch_size)
            
            if batch:
                gen.all_conversations.extend(batch)
                gen.generated += len(batch)
                pbar.update(len(batch))
                
                if gen.generated % CONFIG.checkpoint_every == 0:
                    gen.save_checkpoint(
                        gen.all_conversations[-CONFIG.checkpoint_every:],
                        f"checkpoint_{gen.generated}"
                    )
                    print(f"\nCheckpoint: {gen.generated} pares")
        
        # Split final en 30 archivos
        print(f"\nDividiendo {gen.generated} conversaciones en {CONFIG.num_output_files} archivos...")
        gen.split_into_files(gen.all_conversations)
        
        # Guardado final consolidado
        gen.save_checkpoint(gen.all_conversations, "final")
        
        print(f"\n✅ COMPLETADO: {gen.generated} pares generados")
        print(f"Archivos en: {CONFIG.output_dir}")
        
        return gen.all_conversations

conversations = await main()

In [ ]:
# ============================================================
# VERIFICACIÓN Y SUBIDA A GITHUB
# ============================================================

# Listar archivos generados
import os
output_dir = CONFIG.output_dir
files = sorted(os.listdir(output_dir))
for f in files:
    size = os.path.getsize(os.path.join(output_dir, f)) / 1024
    print(f"{f}: {size:.1f} KB")

# Contar pares totales en archivos chat_XX.txt
total_pairs = 0
for f in files:
    if f.startswith("chat_") and f.endswith(".txt"):
        with open(os.path.join(output_dir, f), "r") as fp:
            content = fp.read()
            pairs = content.count("U: ")
            total_pairs += pairs
            print(f"  {f}: {pairs} pares")
print(f"\nTotal pares U:/B:: {total_pairs}")

In [ ]:
# ============================================================
# SUBIR A GITHUB (rubidium-api/resources/)
# ============================================================

import subprocess

# Configurar git
!git config --global user.email "kaggle@rubidium.ai"
!git config --global user.name "Kaggle Bot"

# Clonar repo si no existe
repo_dir = "/kaggle/working/rubidium-api"
if not os.path.exists(repo_dir):
    !git clone https://github.com/diegovelandiabarajas1-lang/rubidium-api.git {repo_dir}

# Copiar archivos chat_XX.txt a resources/
resources_dir = f"{repo_dir}/resources"
os.makedirs(resources_dir, exist_ok=True)

for f in files:
    if f.startswith("chat_") and f.endswith(".txt"):
        src = os.path.join(output_dir, f)
        dst = os.path.join(resources_dir, f)
        os.system(f"cp {src} {dst}")
        print(f"Copiado: {f}")

# Commit y push
os.chdir(repo_dir)
!git add resources/chat_*.txt
!git commit -m "feat: Add 30 SOC corpus files (500K pairs) chat_06-chat_35" || true
!git push origin main || echo "Push failed - check credentials"